In [ ]:
import os
import rasterio
from rasterio.windows import Window
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
input_data_dsm = '/home/ajai-krishna/work/GEO_AI/data/input/subsets/gujarath_dsm_RGBZ.tif'
input_data_dtm = '/home/ajai-krishna/work/GEO_AI/outputs/gujarath_dtm_RGBZ_filled.tif'
output_dtm_tif = '/home/ajai-krishna/work/GEO_AI/data/Training/dtm_patches'
output_dsm_tif = '/home/ajai-krishna/work/GEO_AI/data/Training/dsm_patches'
# output_trainin

In [ ]:
def extract_patches(data_path, patch_size=256, stride=128, nodata_threshold=0.1, output_dir=None):

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    npy_dir = output_dir / "npy"
    tif_dir = output_dir / "tif"

    npy_dir.mkdir(exist_ok=True)
    tif_dir.mkdir(exist_ok=True)

    with rasterio.open(data_path) as src:

        H = src.height
        W = src.width
        nodata_value = src.nodata
        profile = src.profile.copy()

        patch_id = 0
        skipped = 0

        for y in range(0, H - patch_size + 1, stride):
            for x in range(0, W - patch_size + 1, stride):

                window = Window(x, y, patch_size, patch_size)

                patch = src.read(window=window).astype(np.float32)  # (bands,H,W)

                # nodata ratio check
                nodata_ratio = (patch == nodata_value).mean()

                if nodata_ratio > nodata_threshold:
                    skipped += 1
                    continue

                # ── Save NPY (for ML) ──
                patch_npy = np.transpose(patch, (1,2,0))  # (H,W,bands)

                np.save(npy_dir / f"patch_{patch_id}.npy", patch_npy)

                # ── Save TIF (georeferenced) ──
                transform = src.window_transform(window)

                patch_profile = profile.copy()
                patch_profile.update({
                    "height": patch_size,
                    "width": patch_size,
                    "transform": transform
                })

                tif_path = tif_dir / f"patch_{patch_id}.tif"

                with rasterio.open(tif_path, "w", **patch_profile) as dst:
                    dst.write(patch)

                patch_id += 1

    print(f"Total patches  : {patch_id + skipped}")
    print(f"Saved patches  : {patch_id}")
    print(f"Skipped patches: {skipped}")

In [ ]:
extract_patches(input_data_dsm,patch_size=256,stride=128,nodata_threshold=0.1,output_dir=output_dsm_tif)

In [ ]:
extract_patches(input_data_dtm,patch_size=256,stride=128,nodata_threshold=0.1,output_dir=output_dtm_tif)